In [2]:
# Importação das libs
import pandas as pd
from pandasql import sqldf

In [3]:
pysqldf = lambda q: sqldf(q, globals())

### Leitura dos arquivos

In [15]:
path_dados = r'C:\Users\lfsoares\OneDrive - Sistema FIEMG\Área de Trabalho\LUIZ\3.Python\8.Teste_Engenheiro_de_Dados\dados/'

buyers = pd.read_csv(path_dados + 'buyers.csv', sep=',')
order_items = pd.read_csv(path_dados + 'order_items.csv', sep=',')
orders = pd.read_csv(path_dados + 'orders.csv', sep=',')
payments = pd.read_csv(path_dados + 'payments.csv', sep=',')
products = pd.read_csv(path_dados + 'products.csv', sep=',')
sellers = pd.read_csv(path_dados + 'sellers.csv', sep=',')

In [20]:
print('Tabela ---buyers---')
display(buyers.head())
print('Tabela ---order_items---')
display(order_items.head())
print('Tabela ---orders---')
display(orders.head())
print('Tabela ---payments---')
display(payments.head())
print('Tabela ---products---')
display(products.head())
print('Tabela ---sellers---')
display(sellers.head())


Tabela ---buyers---


,id,name,city,state,segment,created_at
0,1,Nascimento & Costa Distribuidora Mercearia,Recife,DF,supermarket,2023-04-03 03:46:36
1,2,Lima & Almeida Atacado Mercearia,Rio de Janeiro,PE,convenience,2023-08-05 01:36:12
2,3,Ferreira & Costa Grupo Mercearia,São Paulo,PR,grocery,2023-07-10 11:37:01
3,4,Lima & Almeida Comércio Supermercado,Campo Grande,MT,convenience,2023-10-13 01:08:52
4,5,Ferreira & Ferreira Comércio Mercearia,São Paulo,PB,convenience,2024-04-20 00:23:27


Tabela ---order_items---


,id,order_id,product_id,qty,unit_price,discount
0,1,1,500,6,240.06,117.48
1,2,1,778,6,126.47,94.54
2,3,2,346,39,492.08,98.86
3,4,2,248,20,278.55,588.33
4,5,3,701,9,298.75,272.33


Tabela ---orders---


...

,id,order_id,paid_at,amount,method,status
0,1,1,2023-08-23 19:25:59,1987.16,transfer,paid
1,2,2,2024-07-20 17:57:54,24074.93,boleto,paid
2,3,3,2024-06-16 10:16:15,2416.42,transfer,paid
3,4,4,2024-07-11 11:46:30,3871.48,boleto,paid
4,5,5,2023-09-08 00:06:40,15367.65,transfer,paid


Tabela ---products---


,id,name,category,seller_id,active,unit_cost
0,1,Produto Laticínios Linha 1,Snacks,31,1,104.39
1,2,Produto Grãos Linha 2,Carnes,71,1,153.19
2,3,Produto Enlatados Linha 3,Grãos,9,0,90.57
3,4,Produto Snacks Linha 4,Bebidas,13,1,158.39
4,5,Produto Carnes Linha 5,Grãos,58,1,124.43


Tabela ---sellers---


,id,name,state,plan,created_at
0,1,Santos & Silva Atacado Distribuidora,BA,free,2023-04-17 00:16:11
1,2,Souza & Souza Comércio Distribuidora,DF,premium,2022-09-09 22:06:21
2,3,Santos & Almeida Distribuidora Distribuidora,AM,basic,2022-12-16 08:31:25
3,4,Nascimento & Ferreira Alimentos Distribuidora,GO,basic,2023-05-30 23:12:37
4,5,Silva & Santos Suprimentos Distribuidora,BA,premium,2023-03-06 10:29:33


### Desafio 1
Escreva uma query que retorne o faturamento bruto mensal dos últimos 12 meses, 
considerando apenas pedidos com status 'completed' ou 'delivered'. Inclua também a 
quantidade de pedidos e o ticket médio de cada mês. Ordene do mês mais recente ao mais 
antigo.

In [101]:
# converte a coluna created_at para datetime
orders['created_at'] = pd.to_datetime(orders['created_at'])

query = """
    select
        strftime('%Y-%m', created_at ) as anomes
        ,sum(total_value) as faturamento_bruno
        ,count(*) qtd_pedidos
        ,avg(total_value) ticket_medio
    from orders
    where 
        status in ('completed','delivered')
        and date(created_at) >= (select date(max(created_at), '-12 months') from orders)

    group by
        strftime('%Y-%m', created_at )
    order by 1 desc

"""

resultado_desafio1 = pysqldf(query)
resultado_desafio1


,anomes,faturamento_bruno,qtd_pedidos,ticket_medio
0,2024-11,56710238.63,3643,15566.906020
1,2024-10,62493439.13,3863,16177.437000
2,2024-09,60274462.50,3779,15949.844536
3,2024-08,60102539.82,3743,16057.317612
4,2024-07,59769406.95,3862,15476.283519
5,2024-06,58797774.37,3644,16135.503395
6,2024-05,60730315.72,3848,15782.306580
7,2024-04,57779312.59,3653,15816.948423
8,2024-03,61525913.64,3877,15869.464442
9,2024-02,57926376.26,3628,15966.476367


### Desafio 2
A diretora comercial quer um ranking dos 10 sellers com maior crescimento de GMV entre o 
trimestre atual e o anterior, mas só quer ver sellers que tiveram pelo menos 50 pedidos em ambos 
os trimestres (para evitar distorção de sellers novos ou inativos). 
Escreva a query que resolve esse problema, exibindo: nome do seller, estado, GMV do 
trimestre anterior, GMV do trimestre atual e o percentual de crescimento. Ordene pelo maior 
crescimento. 

In [226]:
# Desafio 2
query = """
WITH base AS (
    SELECT
        sellers.id AS id_sellers
        ,sellers.name AS name_sallers
        ,sellers.state AS state_name
        ,strftime('%Y', orders.created_at) AS ano
        ,CASE
            WHEN CAST(strftime('%m', orders.created_at) AS INT) BETWEEN 1 AND 3 THEN 1
            WHEN CAST(strftime('%m', orders.created_at) AS INT) BETWEEN 4 AND 6 THEN 2
            WHEN CAST(strftime('%m', orders.created_at) AS INT) BETWEEN 7 AND 9 THEN 3
            ELSE 4
        END AS trimestre
        ,SUM(orders.total_value) AS gmv
        ,COUNT(*) AS qtd_pedidos
    FROM orders
    JOIN sellers
        ON orders.seller_id = sellers.id
    GROUP BY
        sellers.id
        ,sellers.name
        ,sellers.state
        ,ano
        ,trimestre
),

base_trimestre_anterior AS (
    SELECT
        *
        ,LAG(gmv) OVER ( PARTITION BY id_sellers ORDER BY ano, trimestre ) AS gmv_trimestre_ant
        ,LAG(qtd_pedidos) OVER ( PARTITION BY id_sellers ORDER BY ano, trimestre ) AS qtd_pedidos_trimestre_ant
    FROM base
),

comparacao AS (
    SELECT
        *
        ,((gmv * 1.0 / gmv_trimestre_ant) - 1) * 100 AS variacao_gmv
        ,ROW_NUMBER() OVER ( PARTITION BY id_sellers ORDER BY ano DESC, trimestre DESC) AS ordena_sellers
    FROM base_trimestre_anterior
    WHERE
        gmv_trimestre_ant IS NOT NULL
        AND qtd_pedidos >= 50
        AND qtd_pedidos_trimestre_ant >= 50
)

SELECT
    name_sallers
    ,state_name
    ,gmv_trimestre_ant
    ,gmv AS gmv_trimestre_atual
    ,variacao_gmv
FROM comparacao
WHERE ordena_sellers = 1
ORDER BY variacao_gmv DESC
limit 10;
"""

resultado_desafio2 = pysqldf(query)
resultado_desafio2

,name_sallers,state_name,gmv_trimestre_ant,gmv_trimestre_atual,variacao_gmv
0,Silva & Almeida Alimentos Distribuidora,RN,1028619.01,1373818.54,33.559513
1,Santos & Almeida Comércio Distribuidora,BA,1534199.35,1759286.18,14.671290
2,Ferreira & Oliveira Brasil Distribuidora,ES,1261866.47,1446541.30,14.635053
3,Almeida & Oliveira Comércio Distribuidora,PA,1409468.46,1585930.29,12.519743
4,Souza & Santos Distribuidora Distribuidora,DF,1357385.44,1514441.00,11.570447
5,Silva & Souza Alimentos Distribuidora,BA,1196825.76,1289617.67,7.753168
6,Oliveira & Souza Atacado Distribuidora,ES,1479529.42,1481622.39,0.141462
7,Costa & Ferreira Atacado Distribuidora,PE,977776.85,970702.77,-0.723486
8,Santos & Silva Suprimentos Distribuidora,MS,1186919.92,1116094.58,-5.967154
9,Rodrigues & Almeida Atacado Distribuidora,MG,1185897.83,1108548.84,-6.522399


### Desafio 3
O time de fraudes suspeita que alguns sellers estão aplicando descontos abusivos para inflar 
volume artificialmente. Você precisa encontrar todos os pedidos onde o desconto total (soma dos 
descontos dos itens) representa mais de 40% do valor bruto do pedido, listando também o seller 
responsável e a data do pedido. Exclua pedidos cancelados.

In [246]:
# Desafio 3
query = """
    WITH pedido_itens AS (
        SELECT
            o.id AS order_id
            ,o.seller_id
            ,o.created_at AS data_pedido
            ,SUM(oi.qty * oi.unit_price) AS valor_bruto
            ,SUM(oi.discount) AS desconto_total
        FROM orders o
        JOIN order_items oi
            ON o.id = oi.order_id
        WHERE o.status <> 'cancelled'
        GROUP BY
            o.id
            ,o.seller_id
            ,o.created_at
    )

    SELECT
        pi.order_id
        ,s.name AS seller_name
        ,pi.data_pedido
        ,pi.valor_bruto
        ,pi.desconto_total
        ,(pi.desconto_total * 1.0 / pi.valor_bruto) * 100 AS percentual_desconto
    FROM pedido_itens pi
    JOIN sellers s
        ON pi.seller_id = s.id
    WHERE
        pi.desconto_total > pi.valor_bruto * 0.40
    ORDER BY percentual_desconto DESC
"""

resultado_desafio3 = pysqldf(query)
resultado_desafio3

,order_id,seller_name,data_pedido,valor_bruto,desconto_total,percentual_desconto
0,72401,Costa & Oliveira Atacado Distribuidora,2024-01-21 02:43:43.000000,930.30,558.02,59.982801
1,42513,Costa & Costa Atacado Distribuidora,2024-03-27 02:30:57.000000,4082.60,2447.96,59.960809
2,36119,Costa & Costa Atacado Distribuidora,2024-11-14 01:05:01.000000,21030.24,12598.00,59.904214
3,71165,Santos & Almeida Suprimentos Distribuidora,2024-11-09 02:32:22.000000,13385.60,8017.88,59.899295
4,53223,Costa & Costa Atacado Distribuidora,2024-05-06 01:56:47.000000,7970.40,4773.99,59.896492
...,...,...,...,...,...,...
896,38595,Costa & Costa Atacado Distribuidora,2023-08-21 04:52:02.000000,27785.26,11124.91,40.038891
897,64306,Oliveira & Santos Distribuidora Distribuidora,2024-10-12 21:22:52.000000,6907.95,2765.81,40.038072
898,32057,Souza & Ferreira Alimentos Distribuidora,2024-03-10 03:51:38.000000,24658.56,9868.74,40.021558
899,59315,Santos & Rodrigues Distribuidora Distribuidora,2023-08-10 22:52:09.000000,21647.49,8661.37,40.010967


In [250]:
# Desafio 4
query = """
WITH itens_com_rank AS (
    SELECT
        oi.product_id,
        oi.order_id,
        oi.qty,
        oi.unit_price,
        SUM(oi.qty) OVER (PARTITION BY oi.product_id) AS total_unidades_vendidas,
        MAX(oi.unit_price) OVER (PARTITION BY oi.order_id) AS maior_preco_no_pedido
    FROM order_items oi
)
SELECT
        product_id,
        total_unidades_vendidas
    FROM itens_com_rank
    GROUP BY product_id, total_unidades_vendidas
    HAVING
        total_unidades_vendidas > 1000
        AND MAX(
            CASE
                WHEN unit_price = maior_preco_no_pedido THEN 1
                ELSE 0
            END
        ) = 0


"""

resultado_desafio4 = pysqldf(query)
resultado_desafio4

,product_id,total_unidades_vendidas


Nenhum registro encontrado para a questão número 4

### Observações

Anomalias na base de buyers.
Identificação de buyers duplicados pelo ID

In [278]:
pysqldf("""
WITH base AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY id
            ORDER BY created_at DESC
        ) AS rn
    FROM buyers
),

id_duplicado AS (
    SELECT id
    FROM base
    WHERE rn > 1
)

SELECT
*
FROM base
WHERE id IN (SELECT id FROM id_duplicado);
""")

,id,name,city,state,segment,created_at,rn
0,173,Costa & Santos Brasilis Supermercado,Manaus,RJ,minimarket,2024-03-19 03:02:52,1
1,173,Costa & Santos Brasil Supermercado,Manaus,RJ,minimarket,2024-03-19 03:02:51,2


Anomalias na base de orders.
Identificação de orders duplicados pelo ID

In [280]:
pysqldf("""
WITH base AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY id
            ORDER BY created_at DESC
        ) AS rn
    FROM orders
),

id_duplicado AS (
    SELECT id
    FROM base
    WHERE rn > 1
)

SELECT
*
FROM base
WHERE id IN (SELECT id FROM id_duplicado);
""")

,id,seller_id,buyer_id,status,created_at,total_value,rn
0,79990,79,1070,processing,2024-08-15 19:00:10.000000,12165.99,1
1,79990,79,1070,processing,2024-08-13 19:00:10.000000,12162.98,2
2,79991,4,153,refunded,2024-11-18 09:53:35.000000,1610.87,1
3,79991,4,153,refunded,2024-11-13 09:53:35.000000,1670.88,2
